# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. You'll be guided through obtaining metadata, discovering record sets and fields, loading records, exploring the dataset, and visualizing selected relationships.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset name and description
meta = dataset.metadata
print(meta.name)
print(meta.description)

## 2. Data Overview
Review available record sets (i.e., resource tables), field definitions, and important Croissant `@id` identifiers.

In [ ]:
# List all record_sets, fields, and columns by their @id
record_sets_info = dataset._dataset_jsonld.get('recordSet', []) if hasattr(dataset, '_dataset_jsonld') else []
if not record_sets_info:
    print("No record sets found directly in metadata; trying to discover via dataset api...")

# Record sets can alternatively be obtained through dataset.record_sets (mlcroissant >= v2)
try:
    record_sets = list(dataset.record_sets)
except AttributeError:
    # Fallback to heuristic extraction
    record_sets = []
    if hasattr(dataset, '_dataset_jsonld'):
        rs = dataset._dataset_jsonld.get('recordSet', [])
        if isinstance(rs, list):
            for r in rs:
                if isinstance(r, dict) and '@id' in r:
                    record_sets.append(r)
        elif isinstance(rs, dict) and '@id' in rs:
            record_sets.append(rs)

if not record_sets:
    # Try via metadata (classic for Croissant, e.g. in file downloads)
    if hasattr(dataset.metadata, 'record_sets'):
        record_sets = dataset.metadata.record_sets

print("Available record sets in the dataset:")
record_set_ids = []
for rs in dataset.record_sets:
    print(f"- @id: {rs['@id']}, Name: {rs.get('name', '')}")
    record_set_ids.append(rs['@id'])

# For illustrative purposes, print the fields for each record set
print("\nSample record set fields:")
for rs in dataset.record_sets:
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"Record Set @id: {rs['@id']}, Name: {rs.get('name', '')}")
    for field in fields:
        if isinstance(field, dict):
            print(f"    Field @id: {field.get('@id', '')}, Name: {field.get('name', '')}")
        else:
            print(f"    Field @id: {field}")

## 3. Data Extraction
Load one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from above.

For this dataset, we'll enumerate and extract each available record set by its Croissant `@id`.

In [ ]:
# List record set @ids to extract
all_record_sets = list(dataset.record_sets)
record_set_ids = [rs['@id'] for rs in all_record_sets]

# We'll extract all record sets, but focus on the main table (with the full patient records)
dataframes = {}
for rs_id in record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} rows from record set @id: {rs_id}")

# Display columns for the largest table
largest_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[1] if not dataframes[k].empty else 0)
print(f"\nColumns available in record set @id: {largest_rs_id}")
print(dataframes[largest_rs_id].columns.tolist())
dataframes[largest_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main clinical table, select a numeric field, and demonstrate data filtering and normalization.

**Note:** All fields/columns are referenced by their Croissant `@id`.

In [ ]:
# Choose main clinical table record set @id
main_rs_id = largest_rs_id  # from previous extraction
df = dataframes[main_rs_id].copy()
print(f"Analyzing clinical table with shape: {df.shape}")

# Display all columns with their @id
print("\nAvailable columns (@id):")
for i, c in enumerate(df.columns):
    print(f"{i+1:2d}. {c}")

# Identify a numeric field by @id; assuming Age is available (update as per true @id)
age_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        age_field_id = col
        break

if not age_field_id:
    print("No age field automatically detected. Please update field id manually.")
    age_field_id = df.columns[0]  # fallback for demo

# Filter for patients older than 50
threshold = 50
is_numeric = pd.api.types.is_numeric_dtype(df[age_field_id])
if not is_numeric:
    df[age_field_id] = pd.to_numeric(df[age_field_id], errors='coerce')

filtered_df = df[df[age_field_id] > threshold]
print(f"\nFiltered records with {age_field_id} > {threshold} (N={filtered_df.shape[0]}):")
print(filtered_df[[age_field_id]].head())

# Normalize the numeric field
filtered_df[f"{age_field_id}_normalized"] = (
    filtered_df[age_field_id] - filtered_df[age_field_id].mean()
) / filtered_df[age_field_id].std()
print(f"\nNormalized {age_field_id} for filtered records:")
print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

# Grouping: Try grouping by 'Sex' if available
group_field_id = None
for col in df.columns:
    if 'sex' in col.lower() or 'gender' in col.lower():
        group_field_id = col
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[age_field_id].mean().reset_index()
    print(f"\nMean {age_field_id} grouped by {group_field_id}:")
    print(grouped)

## 5. Visualization
Visualize the distribution of the selected numeric field (e.g., age) and compare across grouping (e.g., by Sex/Gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if age_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[age_field_id], bins=15, kde=True)
    plt.title(f'Distribution of {age_field_id}')
    plt.xlabel('Age')
    plt.ylabel('Count')
    plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[group_field_id], y=df[age_field_id])
    plt.title(f'{age_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(age_field_id)
    plt.show()

## 6. Conclusion
In this notebook, you have:
- Loaded FAIR^2 clinical oncology data using its Croissant schema with the `mlcroissant` library,
- Identified the available record sets and fields by their `@id`,
- Extracted the primary clinical table and performed filtering and normalization by a numeric (age) field,
- Grouped and visualized data by key attributes.

This streamlined workflow, rooted in explicit Croissant data modeling and robust record set referencing by `@id`, is ideally suited for FAIR biomedical and clinical data science, enabling transparent, reproducible EDA.
